In [2]:
import os
from pathlib import Path
import pandas as pd
import wget
import matplotlib.pyplot as plt
import numpy as np

from atomworks.io import parse
from atomworks.io.utils.testing import get_pdb_path_or_buffer
from atomworks.io.utils.visualize import view


In [18]:
df = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate.csv")
stoichiometry_df = pd.read_csv(Path.home() / "data/bad_afdb/stoichiometry_results_full.csv")
print(df.columns)
print(stoichiometry_df.columns)
df = pd.merge(df, stoichiometry_df, on="pdb", how="inner")
print(f"Full dataset: {len(df)}")
df = df[df["stoichiometry"] == "A1"]
print(f"After stoichiometry filter: {len(df)}")

df.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer.csv", index=False)
df["pdb"].describe()


Index(['pdb', 'uniprot', 'identity', 'plddt', 'coverage', 'length'], dtype='object')
Index(['pdb', 'stoichiometry', 'oligomeric_state', 'full_composition'], dtype='object')
Full dataset: 25871
After stoichiometry filter: 5269


count       5269
unique      5269
top       5QBC_A
freq           1
Name: pdb, dtype: object

In [ ]:
import subprocess
from tqdm import tqdm

df = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer.csv")
results_df = pd.read_csv(Path.home() / "data/bad_afdb/pdb_vs_afdb_tm_scores.csv")

# Directories
pdb_dir = Path.home() / "data/bad_afdb/pdb"
afdb_dir = Path.home() / "data/bad_afdb/afdb"
afdb_dir.mkdir(parents=True, exist_ok=True)

# Results storage
results = []

# Process each row
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    pdb_chain_id = row["pdb"]
    if pdb_chain_id in results_df["pdb"].values:
        continue
    uniprot_id = row["uniprot"]
    pdb_id, chain_id = pdb_chain_id.split("_")
    
    # PDB file path (middle two chars as subdir)
    pdb_subdir = pdb_dir / pdb_id[1:3]
    pdb_file = pdb_subdir / f"{pdb_id}.cif"
    
    # Download PDB if needed
    if not pdb_file.exists():
        pdb_subdir.mkdir(parents=True, exist_ok=True)
        subprocess.run(["wget", "-q", f"https://files.rcsb.org/download/{pdb_id}.cif", "-O", str(pdb_file)])
    
    # AFDB file path (using v6 - current version)
    afdb_file = afdb_dir / f"AF-{uniprot_id}-F1-model_v6.cif"
    
    # Download AFDB if needed
    if not afdb_file.exists():
        url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v6.cif"
        result = subprocess.run(["wget", "-q", url, "-O", str(afdb_file)], capture_output=True)
        if result.returncode != 0:
            print(f"Failed to download AFDB for {uniprot_id}")
            results.append({
                "pdb_chain": pdb_chain_id,
                "uniprot": uniprot_id,
                "tm_score": None,
                "error": "AFDB download failed"
            })
            continue
    
    # Check files exist
    if not pdb_file.exists():
        results.append({
            "pdb_chain": pdb_chain_id,
            "uniprot": uniprot_id,
            "tm_score": None,
            "error": "PDB file missing"
        })
        continue
    
    if not afdb_file.exists() or afdb_file.stat().st_size == 0:
        results.append({
            "pdb_chain": pdb_chain_id,
            "uniprot": uniprot_id,
            "tm_score": None,
            "error": "AFDB file missing or empty"
        })
        continue
    
    # Run USalign: PDB chain vs AFDB (TM normalized by PDB chain length = TM1)
    # USalign <pdb> -chain1 <chain> <afdb> -outfmt 2
    cmd = [
        "USalign",
        str(pdb_file),
        "-chain1", chain_id,
        str(afdb_file),
        "-outfmt", "2",
        "-TMscore", "5",
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        results.append({
            "pdb": pdb_chain_id,
            "uniprot": uniprot_id,
            "tm_score": None,
            "error": f"USalign failed: {result.stderr[:100]}"
        })
        continue
    
    # Parse USalign output for TM1 (normalized by first structure = PDB chain)
    tm_score = None
    for line in result.stdout.strip().split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        parts = line.split("\t")
        if len(parts) >= 3:
            tm_score = float(parts[2])  # TM1 column
            break
    
    results.append({
        "pdb": pdb_chain_id,
        "uniprot": uniprot_id,
        "tm_score": tm_score,
        "error": None
    })

# Create results dataframe
results_df = pd.concat([results_df, pd.DataFrame(results)])
results_df.to_csv(Path.home() / "data/bad_afdb/pdb_vs_afdb_tm_scores.csv", index=False)

# Summary
print(f"Total: {len(results_df)}")
print(f"Successful: {results_df['tm_score'].notna().sum()}")
print(f"Failed: {results_df['tm_score'].isna().sum()}")
print(f"\nTM-score statistics:")
display(results_df.describe())
display(results_df.head())


In [20]:
# Create results dataframe
results_df = pd.concat([results_df, pd.DataFrame(results)])
results_df.to_csv(Path.home() / "data/bad_afdb/pdb_vs_afdb_tm_scores.csv", index=False)

# Summary
print(f"Total: {len(results_df)}")
print(f"Successful: {results_df['tm_score'].notna().sum()}")
print(f"Failed: {results_df['tm_score'].isna().sum()}")
print(f"\nTM-score statistics:")
display(results_df.describe())
display(results_df.head())


Total: 5269
Successful: 5131
Failed: 138

TM-score statistics:


,tm_score
count,5131.000000
mean,0.892054
std,0.171697
min,0.021000
25%,0.894300
50%,0.963700
75%,0.985600
max,0.999700


,pdb,uniprot,tm_score,error,pdb_chain
0,5QBC_A,P11838,0.9989,NaN,NaN
1,5QCS_B,P56817,0.9916,NaN,NaN
2,5QIX_A,Q96DC9,0.9798,NaN,NaN
3,5QKU_A,P0AEG4,0.9662,NaN,NaN
4,5QTY_A,P03951,0.9827,NaN,NaN


In [26]:
df = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer.csv")
results_df = pd.read_csv(Path.home() / "data/bad_afdb/pdb_vs_afdb_tm_scores.csv")
df_tm = pd.merge(df, results_df[["pdb", "tm_score"]], on="pdb", how="inner")

df_tm.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_tm.csv", index=False)
display(df_tm.head())
display(df_tm.describe())


,pdb,uniprot,identity,plddt,coverage,length,stoichiometry,oligomeric_state,full_composition,tm_score
0,5QBC_A,P11838,1.000000,97.720545,0.787589,330,A1,Monomer,['A1'],0.9989
1,5QCS_B,P56817,1.000000,96.857082,0.752495,377,A1,Monomer,['A1'],0.9916
2,5QIX_A,Q96DC9,0.995516,96.124933,0.952991,223,A1,Monomer,['A1'],0.9798
3,5QKU_A,P0AEG4,1.000000,96.994947,0.903846,188,A1,Monomer,['A1'],0.9662
4,5QTY_A,P03951,0.991597,86.544076,0.380800,238,A1,Monomer,['A1'],0.9827


,identity,plddt,coverage,length,tm_score
count,5242.000000,5242.000000,5242.000000,5242.000000,5131.000000
mean,0.926149,90.848006,0.677961,299.932278,0.892054
std,0.232500,8.978549,0.308353,223.007093,0.171697
min,0.000000,28.618261,0.004971,2.000000,0.021000
25%,0.990050,88.957414,0.404876,147.000000,0.894300
50%,1.000000,93.565911,0.818182,263.000000,0.963700
75%,1.000000,96.264772,0.945141,391.000000,0.985600
max,1.000000,98.753545,1.000000,2302.000000,0.999700


In [27]:
df_tm = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_tm.csv")
df_tm = df_tm[df_tm["tm_score"] < 0.5]

display(df_tm.describe())
display(df_tm.head())
df_tm.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_tm-05.csv", index=False)


,identity,plddt,coverage,length,tm_score
count,274.000000,274.000000,274.000000,274.000000,274.000000
mean,0.840626,68.770035,0.407705,93.332117,0.323082
std,0.283602,17.908567,0.340693,156.584876,0.121083
min,0.031250,28.618261,0.007317,3.000000,0.021000
25%,0.857143,55.114276,0.108222,20.000000,0.212425
50%,0.986498,71.293974,0.285079,39.000000,0.336100
75%,1.000000,83.778892,0.718856,112.000000,0.436400
max,1.000000,98.411563,0.996124,1523.000000,0.498300


,pdb,uniprot,identity,plddt,coverage,length,stoichiometry,oligomeric_state,full_composition,tm_score
29,6GIG_A,P0DUM4,1.000000,81.286897,0.966667,29,A1,Monomer,['A1'],0.4248
269,6SLO_A,Q9UHX1,0.504673,61.749206,0.382826,214,A1,Monomer,['A1'],0.4777
315,6U6G_A,A0A396GSL0,1.000000,73.942778,0.529412,36,A1,Monomer,['A1'],0.3260
319,6UF2_A,Q31PX7,1.000000,93.797903,0.992000,124,A1,Monomer,['A1'],0.4655
348,6VLA_A,Q9H0A0,1.000000,90.417647,0.016585,17,A1,Monomer,['A1'],0.4272


In [ ]:
# Length filter on the TRUE structure length: pdb_length = model-1 cif-chain CA count (the length the
# model actually samples), NOT the AFDB/UniProt monomer `length` column. The two diverge for proteins
# deposited as a larger construct (e.g. 6M5Y monomer-132/chain-277, 8XHT monomer-100/chain-287).
# Applied right after tm-05 (cifs were downloaded during the TM step) and BEFORE coverage/identity,
# so the output filename order matches the order the filters are applied.
pdb_dir = Path.home() / "data/bad_afdb/pdb"

def _chain_model1_ca(cif_path, chain):
    n = 0
    with open(cif_path) as f:
        for line in f:
            if not line.startswith("ATOM"):
                continue
            p = line.split()  # mmCIF atom_site: p[3]=label_atom_id, p[6]=label_asym_id, p[-1]=model_num
            if len(p) >= 8 and p[3] == "CA" and p[6] == chain and p[-1] == "1":
                n += 1
    return n

def _pdb_length(pdb_chain):
    pid, chain = str(pdb_chain).split("_", 1)
    cif = pdb_dir / pid[1:3] / f"{pid}.cif"
    return _chain_model1_ca(str(cif), chain) if cif.exists() else None

df_tm = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_tm-05.csv")
df_tm["pdb_length"] = df_tm["pdb"].map(_pdb_length)
print(f"Before length filter: {len(df_tm)}  (pdb_length NaN: {df_tm['pdb_length'].isna().sum()})")
df_tm = df_tm[(df_tm["pdb_length"] >= 50) & (df_tm["pdb_length"] <= 512)]
print(f"After pdb_length 50-512 filter: {len(df_tm)}")

display(df_tm.describe())
display(df_tm.head())
df_tm.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_tm-05_pdbLength_50-512.csv", index=False)

In [ ]:
df_tm = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_tm-05_pdbLength_50-512.csv")
df_tm = df_tm[df_tm["coverage"] >= 0.8]
df_tm = df_tm[df_tm["identity"] >= 0.7]

display(df_tm.describe())
display(df_tm.head())
df_tm.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_tm-05_pdbLength_50-512_coverage-08_identity-07.csv", index=False)

In [ ]:
# Leakage filter: drop targets released before the model's training cutoff.
# 2019-08-28 = the training cutoff for BOTH the AF2 non-multimer weights and the
# proteina sequence finetune; a structure released before it could be in training.
# (For this bad-AFDB set every target is post-2020, so this is currently a no-op.)
from fetch_release_dates import fetch_release_dates

RELEASE_CUTOFF = "2019-08-28"
df_dated = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_tm-05_pdbLength_50-512_coverage-08_identity-07.csv")
pdb4 = df_dated["pdb"].astype(str).str.split("_").str[0].str.upper()
df_dated["release_date"] = pdb4.map(fetch_release_dates(sorted(set(pdb4))))
pre = df_dated[df_dated["release_date"] < RELEASE_CUTOFF]
print(f"Dropping {len(pre)} pre-{RELEASE_CUTOFF} (potential leakage) targets: {sorted(pre['pdb'])}")
df_clean = df_dated[df_dated["release_date"] >= RELEASE_CUTOFF].reset_index(drop=True)
print(f"Kept {len(df_clean)} / {len(df_dated)} post-cutoff targets")
df_clean.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_tm-05_pdbLength_50-512_coverage-08_identity-07_releaseFiltered.csv", index=False)
display(df_clean[["pdb", "length", "pdb_length", "tm_score", "release_date"]])

## Old

In [ ]:
df = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_plddt_50_length_50-512.csv")
pdb_dir = Path.home() / "data/bad_afdb/pdb"
pdb_dir.mkdir(parents=True, exist_ok=True)

for index, row in df.iterrows():
    pdb_chain_id = row["pdb"]
    pdb_id, chain_id = pdb_chain_id.split("_")
    pdb_subdir = pdb_dir / pdb_id[1:3]
    pdb_subdir.mkdir(parents=True, exist_ok=True)
    pdb_file = pdb_subdir / f"{pdb_id}.cif"
    if not pdb_file.exists():
        wget.download(f"https://files.rcsb.org/download/{pdb_id}.cif", str(pdb_file))
    # result_dict = parse(
    #     filename=get_pdb_path_or_buffer(pdb_id),
    #     build_assembly=["1"],
    #     add_missing_atoms=True,
    #     remove_waters=True,
    #     hydrogen_policy="remove",
    #     model=1,
    # )
    # print("Keys in parsed result:", list(result_dict.keys()))
    # asym_unit = result_dict["asym_unit"][0]
    # asym_unit = asym_unit[asym_unit.occupancy > 0]
    # view(asym_unit)
    # assembly = result_dict["assemblies"]["1"][0]
    # assembly = assembly[assembly.occupancy > 0]
    # view(assembly)

    # print("Chain info:", result_dict["chain_info"])
    # print("Ligand info:", result_dict["ligand_info"])
    # print("Metadata:", result_dict["metadata"])
    # print("Annotation categories:", result_dict["asym_unit"][0].get_annotation_categories())

    # for chain_id, info in result_dict["chain_info"].items():
        # print(info.keys())
        # print(chain_id, info["processed_entity_canonical_sequence"])


In [7]:
cross_protein_summary_path = Path.home() / "proteina/inference/inference_seq_cond_sampling_finetune-all_8-seq_purge-7bny-7kww-7ad5-7b76_045-noise/bad_afdb_cross_protein_analysis/cross_protein_summary_data.csv"
inference_data_path = Path.home() / "proteina/inference/inference_seq_cond_sampling_finetune-all_8-seq_purge-7bny-7kww-7ad5-7b76_045-noise/"
cross_protein_summary_data = pd.read_csv(cross_protein_summary_path)
print(cross_protein_summary_data.columns)
cross_protein_summary_data["tm_gain"] = cross_protein_summary_data["tm_ref_pred_at_max_ptm"] - cross_protein_summary_data["tms_msa"]
cross_protein_summary_data.sort_values(by="tm_gain", ascending=False, inplace=True)
print(cross_protein_summary_data[cross_protein_summary_data["in_train"] == False].head(10))


Index(['protein_id', 'spearman_rho_composite', 'spearman_rho_ptm',
       'max_tm_ref_template', 'max_tm_ref_pred',
       'tm_ref_template_at_max_composite', 'tm_ref_pred_at_max_composite',
       'tm_ref_pred_at_max_ptm', 'top_1_tm_ref_template',
       'top_5_tm_ref_template', 'top_1_tm_ref_pred', 'top_5_tm_ref_pred',
       'tms_msa', 'in_train', 'length'],
      dtype='object')
    protein_id  spearman_rho_composite  spearman_rho_ptm  max_tm_ref_template  \
92      7ZK0_A                0.741674          0.637971              0.68552   
81      7ZKD_A                0.189965          0.286865              0.52094   
19      6TF4_A                0.403211          0.244504              0.65449   
136     8ZWZ_A                0.460202          0.531158              0.75073   
33      8JB9_A                0.466143          0.448983              0.62478   
50     7N6G_1Q                0.344378          0.420013              0.46978   
11      8HTU_K                0.496953         

In [9]:
cross_protein_summary_data.sort_values(by="tm_ref_pred_at_max_ptm", ascending=True, inplace=True)
print(cross_protein_summary_data.head(10))

    protein_id  spearman_rho_composite  spearman_rho_ptm  max_tm_ref_template  \
78     8GLV_7I                0.116014         -0.000292              0.12802   
39     8GLV_7P               -0.156837         -0.122844              0.18302   
79     9E2G_3T               -0.003219          0.002931              0.15069   
29     9E78_2O                0.061748          0.057508              0.18393   
41     8I7O_E2                0.053322         -0.045574              0.19555   
109    8GZU_a7               -0.082717         -0.461497              0.18836   
53      7KWZ_A               -0.062996         -0.143254              0.19512   
103    8GZU_0E               -0.028757         -0.154505              0.18743   
116     8B12_X                0.152397          0.257148              0.21421   
63      8YVE_X                0.175820          0.101330              0.20455   

     max_tm_ref_pred  tm_ref_template_at_max_composite  \
78           0.12711                           0.0

In [2]:
pdb_file = str(Path.home() / "data/cath_v4-4/by_fold/3.30.70/5k8mA02")
result_dict = parse(filename=pdb_file, file_type="pdb", build_assembly=["1"], add_missing_atoms=True, remove_waters=True, hydrogen_policy="remove", model=1)
print(f">5k8mA02\n{result_dict['chain_info']['A']['processed_entity_canonical_sequence']}")

pdb_file = str(Path.home() / "data/7ad5_example/7ad5.pdb")
result_dict = parse(filename=pdb_file, file_type="pdb", build_assembly=["1"], add_missing_atoms=True, remove_waters=True, hydrogen_policy="remove", model=1)
print(f">7ad5\n{result_dict['chain_info']['A']['processed_entity_canonical_sequence']}")


/home/jupyter-chenxi/.conda/envs/atomworks/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:470: UserWarning: 1035 elements were guessed from atom name
  warnings.warn(
Chain A contains both polymer and non-polymer residues; separating them for processing, naming the non-polymer residues as B.
Chain A contains both polymer and non-polymer residues; separating them for processing, naming the non-polymer residues as B.


>5k8mA02
EAGKVFLKVTVFAGPSRYTPYVPRPVAALDSPNAIVRAKLVEALEEWADNYEKRYTREYGGGTVVPKVAIGAIRGGVPYKIYRFPELCSIYDIRLNPDTNPLVVQREVEAVVSKLGLKAEVKPFLFRRG


No suitable coordinates found for 'NI' among preferences ('ideal_pdbx', 'model', 'ideal_rdkit'). Coordinates will be 'nan'.


>7ad5
GHMHDCHQVTVSRDVTLQNKERHDCNQVCASIDKETENKLNTDIIPRLTRYMSVKGNSIIARVQQSNSDPKCSCTWRAIIWRVYKAYDENSLNVALHVSHPNQQIGENPDWSLVISNPNVHCLK
